# **Zarządzanie konfiguracją w Pythonie**

# **1. Czym jest konfiguracja aplikacji?**

Konfiguracja aplikacji to zestaw parametrów określających sposób działania programu, które nie są częścią jego logiki biznesowej. Innymi słowy, kod odpowiada za to, co aplikacja robi, a konfiguracja za to, jak działa.

Dzięki konfiguracji możliwe jest łatwe zmienianie ustawień, takich jak port serwera, tryb debugowania, dane do bazy czy ścieżki do plików, bez konieczności modyfikowania kodu. Zwiększa to elastyczność aplikacji i pozwala dostosować ją do różnych środowisk, np. testowego czy produkcyjnego.

Konfiguracja może pochodzić z różnych źródeł, takich jak pliki (np. YAML), argumenty linii poleceń czy wartości domyślne w kodzie. Zazwyczaj obowiązuje zasada priorytetu: argumenty uruchomienia nadpisują dane z pliku, a te z kolei nadpisują wartości domyślne.

Dobrze zaprojektowana konfiguracja jest czytelna, podzielona na logiczne sekcje i oddzielona od logiki aplikacji. Dzięki temu kod jest bardziej przejrzysty, łatwiejszy w utrzymaniu i mniej podatny na błędy.


# **2. Format YAML jako plik konfiguracyjny**

Jednym z najczęściej używanych formatów konfiguracji jest YAML.

Cechy YAML:
- czytelny dla człowieka,
- wspiera zagnieżdżone struktury,
- mapuje się bezpośrednio na struktury Pythona.

Przykład:

```
server:
  host: "127.0.0.1"
  port: 8080
```
Po wczytaniu do Pythona:

```
{
  "server": {
    "host": "127.0.0.1",
    "port": 8080
  }
}
```
YAML → Python = dict, list, str, int, float, bool

Oficjalna dokumentacja YAML: https://yaml.org/spec/

Dokumentacja PyYAML (najczęściej używana w Pythonie): https://pyyaml.org/wiki/PyYAMLDocumentation


# **3. Parsowanie argumentów programu — argparse**

Programy konsolowe często przyjmują argumenty z terminala.

Przykład:

```
python main.py --config config.yaml --port 9000
```
Do ich obsługi używamy biblioteki argparse.
Oficjalna dokumentacja argparse: https://docs.python.org/3/library/argparse.html

Kluczowe elementy:
```
parser.add_argument("--port", type=int)
parser.add_argument("--config", type=Path)
parser.add_argument("--debug", action="store_true")
```
Dzięki type możemy automatycznie konwertować dane do:

- int
- float
- str
- Path

pathlib (dla type=Path): https://docs.python.org/3/library/pathlib.html


# **4. Łączenie wielu źródeł konfiguracji**

W praktyce konfiguracja pochodzi z różnych źródeł:

- wartości domyślne (w kodzie),
- plik konfiguracyjny (YAML),
- argumenty z terminala (CLI).

Priorytet (bardzo ważne):

```
CLI > plik YAML > wartości domyślne
```
To oznacza:

- użytkownik zawsze może nadpisać konfigurację przy uruchomieniu programu,
- plik konfiguracyjny nadpisuje domyślne ustawienia,
- brakujące wartości są uzupełniane domyślnymi.


# **5. Struktury danych w konfiguracji**

W konfiguracji używamy różnych typów danych:
| Typ     | Zastosowanie        |
| ------- | ------------------- |
| `int`   | porty, liczby       |
| `float` | timeouty            |
| `str`   | nazwy               |
| `Path`  | ścieżki             |
| `dict`  | zagnieżdżone sekcje |
| `tuple` | dane uporządkowane  |
| `set`   | unikalne wartości   |

Szczególnie ważny:

- set — usuwa duplikaty,
- Path — lepszy niż zwykły string dla ścieżek.

# **6. Problem: dlaczego sam dict to za mało?**

Na początku konfiguracja jest przechowywana w słowniku:

```
config["server"]["port"]
```
Problemy:

- brak typowania,
- brak kontroli nad strukturą,
- łatwo o błędy (literówki),
- brak walidacji.

Rozwiązanie: przejście na obiekty (dataclass)

Oficjalna dokumentacja dataclasses (Python): https://docs.python.org/3/library/dataclasses.html

# **7. Dataclass — uporządkowana konfiguracja**

dataclass to sposób definiowania klas przechowujących dane.

Przykład:

```
from dataclasses import dataclass

@dataclass
class ServerConfig:
    host: str
    port: int
```
Zalety:

- czytelna struktura,
- typowanie,
- automatyczne metody (__init__, __repr__).


# **8. Podział konfiguracji na sekcje**

Zamiast jednego dużego słownika:


```
config = {...}
```
tworzymy strukturę:

```
GlobalConfig(
    app=AppConfig(...),
    server=ServerConfig(...),
    database=DatabaseConfig(...)
)
```

Każda sekcja:
- ma swoją klasę,
- odpowiada jednemu obszarowi systemu.



# **9. Metoda from_dict() — konwersja danych**

Dane z YAML są słownikiem → trzeba je przekształcić w obiekty.

Dlatego stosujemy:

```
@classmethod
def from_dict(cls, data):
    return cls(...)
```

Rola from_dict():

- odczyt danych,
- ustawienie wartości domyślnych,
- konwersja typów (np. Path, set),
- walidacja.

To jest fundament „fabryki configa”.

# **10. Fabryka konfiguracji**

Proces budowy konfiguracji:


1.   wczytaj YAML → dict,
2.   sparsuj CLI,
3.   połącz dane (z priorytetem),
4.   przekaż do from_dict,
5.   utwórz GlobalConfig.

To działa jak fabryka obiektów.

# **11. Globalny config i singleton**

W aplikacji chcemy mieć jedną konfigurację:


```
config = get_config()
```
Dlaczego?

- unikamy duplikacji,
- wszystkie moduły korzystają z tych samych danych,
- spójność systemu.

Singleton — jedna instancja

Implementujemy:
- setup_config(...) → tworzy config raz
- get_config() → zwraca config

Po utworzeniu:

- config istnieje przez cały czas działania programu,
- nie można go ponownie stworzyć.

Moduły w Pythonie: https://docs.python.org/3/tutorial/modules.html

# **12. Serializacja — metoda to_dict()**

Aby zapisać konfigurację do pliku, trzeba ją przekształcić do słownika.

```
def to_dict(self):
    return {...}
```
Ważne konwersje:
| Typ     | Do zapisu |
| ------- | --------- |
| `Path`  | `str`     |
| `set`   | `list`    |
| `tuple` | `list`    |


# **13. Zapisywanie konfiguracji do plików**

Na podstawie dict możemy zapisać konfigurację do:

- YAML
- JSON
- TOML

Dzięki temu:

- można eksportować konfigurację,
- debugować system,
- tworzyć nowe pliki konfiguracyjne.